In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import h5py
import matplotlib.pyplot as plt
import numpy as np
import scipy.signal

import brighteyes_flim.tools_phasor as brighteyes_flim
import brighteyes_ism.analysis.Graph_lib as gr
import brighteyes_ism.dataio.mcs as mcs
from scipy.ndimage import shift
from scipy.optimize import curve_fit
from skimage.filters import gaussian
from skimage.registration import phase_cross_correlation


## Confocal processing

### Open the dataset 4D (x,y,t,ch) of the TTM's IRF

In [ ]:
h = h5py.File(r"C:\Users\REPLACE_ME\Desktop\images\Gold_beads_17_luglio_2024_TTM_IRF\gold_beads_4d","r")
print(h.keys())
img = h["image_4d"]
print(img.shape)

In [ ]:
plt.figure()
img_hist = np.sum(img, axis=(0,1))
for j in range(img_hist.shape[-1]):
    plt.plot(img_hist[:, j])

### Clean IRF

In [ ]:
nch = 25
print("nchannels", nch)

shift_vec = np.empty( nch )

for i in range(nch):
    shift_vec[i], *_ = phase_cross_correlation(img_hist[:, 12], img_hist[:, i], upsample_factor=10, normalization=None)


hist_shifted = np.empty_like(img_hist)
for i in range(nch):
    hist_shifted[:, i] = shift(img_hist[:, i], shift_vec[i], order = 1, mode='grid-wrap')

In [ ]:
plt.figure()
for i in range(hist_shifted.shape[-1]):
    plt.plot(hist_shifted[:, i])


In [ ]:
print(hist_shifted.shape)
irf_sum = np.sum(hist_shifted, axis = -1)
plt.figure()
plt.plot(irf_sum)

In [ ]:
offset = np.mean(irf_sum[0:150])
print(offset)
irf_without_offset = irf_sum - offset
plt.figure()
plt.plot(irf_without_offset)

In [ ]:
def centroid(data):
    x = np.arange(data.size)
    idx = np.sum(x*data)/data.sum()
    return idx

def clean_irf(irf, threshold = 0.3, window = 6):
    
    time = np.arange(irf.size)
    
    t_irf = np.where(irf>threshold, irf, 0)
    
    t0 = centroid(t_irf)
    
    indices = np.argwhere(np.logical_and(time > t0 - window, time < t0 + window))
    
    final_irf = np.zeros_like(irf)
    
    final_irf[indices] = irf[indices]
    
    return final_irf

final_irf = np.empty_like(irf_without_offset)
dt = 0.048
time = np.arange(260) * dt

#for n in range(img_hist.shape[-1]):

irf = irf_without_offset
 
final_irf = clean_irf(irf, threshold = 3000, window = 2/dt)
final_irf = np.where(final_irf>0, final_irf, 0)    
plt.figure()
plt.plot(time, irf)
plt.plot(time, final_irf)

### Open the raw 4D image's histograms not calibrated

In [ ]:
h_1 = h5py.File(r"C:\Users\REPLACE_ME\Desktop\images\Convallaria_29_luglio_2024\convallaria_LP53_500us_50um_300x300_Raw_4D","r")
print(h_1.keys())
img_data = h_1["image_4d"]
print(img_data.shape)

In [ ]:
hist = np.sum(img_data, axis=(0,1))
plt.figure()
for i in range(img_data.shape[-1]):
    plt.plot(hist[:, i])

### Realign temporally the histograms of each pixel using the same shift vectors of the IRF

In [ ]:
nch = img_data.shape[-1]
dset_shifted = np.empty_like(img_data)
shift_dset = np.zeros( (img_data.ndim - 1, nch) )
shift_dset[-1] = shift_vec
print("shift_dset", shift_dset.shape)

for i in range(nch):
    dset_shifted[..., i] = shift(img_data[..., i], shift_dset[:, i], order = 1, mode='grid-wrap')
    
print("dset_shifted", dset_shifted.shape)

data_3D = dset_shifted.sum(-1)

In [ ]:
data_2D = np.sum(data_3D, axis=-1)
print(data_2D.shape)
plt.figure()
plt.hist(data_2D.flatten(), bins = 100, range = (0, 1000))

### Phasor plot of the histograms before calibration

In [ ]:
phasor_matrix_1= brighteyes_flim.phasor(data_3D)
print(phasor_matrix_1.shape)
brighteyes_flim.plot_phasor(phasor_matrix_1[:], bins_2dplot=200, log_scale=False, quadrant='all', dfd_freq = 80e6)

### Phasor plot of the histograms after calibration

In [ ]:
phasor_irf = brighteyes_flim.phasor(final_irf)
print(phasor_irf)

In [ ]:
phasor_cleaned = phasor_matrix_1 / phasor_irf
brighteyes_flim.plot_phasor(phasor_cleaned, bins_2dplot=200, log_scale=False, quadrant='first', dfd_freq = 80e6)

In [ ]:
tau_m_clean = brighteyes_flim.calculate_tau_m(phasor_cleaned, dfd_freq = 80e6)
tau_m_data = 1e9*tau_m_clean.flatten()

plt.figure()
plt.hist(tau_m_data, range = (0, 5), bins = 50)

In [ ]:
tau_phi = brighteyes_flim.calculate_tau_phi(phasor_cleaned, dfd_freq = 80e6)
tau_data = 1e9*tau_phi.flatten()

plt.figure()
plt.hist(tau_data, range = (0, 5), bins = 50)

In [ ]:
hist_indexes = np.argwhere(data_2D > 70)
tau_phi_denoised = tau_phi[hist_indexes[:, 0], hist_indexes[:, 1]]
print(tau_phi_denoised)
plt.figure()
plt.hist(1e9*tau_phi_denoised, bins = 100, range = (-1, 10))

In [ ]:
tau_m_denoised = tau_m_clean[hist_indexes[:, 0], hist_indexes[:, 1]]
print(tau_m_denoised.shape)
plt.figure()
plt.hist(1e9*tau_m_denoised, bins = 100, range = (-1, 10))

### APR processing

In [ ]:
import brighteyes_ism.analysis.APR_lib as apr
from skimage.registration import phase_cross_correlation
from scipy.ndimage import shift
from tqdm import tqdm

In [ ]:
print(dset_shifted.shape)
image_3d = np.sum(dset_shifted, axis = -2)
print(image_3d.shape)
shift_vectors, err = apr.ShiftVectors(image_3d, usf = 100, ref = 12)
gr.PlotShiftVectors(shift_vectors)
print (shift_vectors)     

In [ ]:
with h5py.File(r"C:\Users\REPLACE_ME\Desktop\images\Convallaria_29_luglio_2024\APR_Convallaria_500us", 'w') as f:
     x_size, y_size, bin_size, channel_size = dset_shifted.shape[0], dset_shifted.shape[1], dset_shifted.shape[2], dset_shifted.shape[3]
# Create an empty dataset with dimensions (x,y,t, ch)

     dataset_shape = (x_size, y_size, bin_size, channel_size)
     h5_dataset = f.create_dataset('data', shape=dataset_shape, dtype=np.uint64)
    

     for bin in tqdm(range(dset_shifted.shape[-2])):
         h5_dataset[:, :, bin, :] = apr.Reassignment(shift_vectors, dset_shifted[:, :, bin, :], mode = 'interp')

In [ ]:
f = h5py.File(r"C:\Users\REPLACE_ME\Desktop\images\Convallaria_29_luglio_2024\APR_Convallaria_500us", 'r')
h5_dataset = f["data"]
print(h5_dataset.shape)

In [ ]:
h5_dataset_sum = np.sum(h5_dataset, axis=-1)

In [ ]:
data_histograms = np.sum(h5_dataset_sum, axis = -1)
print(data_histograms.shape)
    
# Plot the histogram of the photon counts in each pixel to see the distribution (e.g. check the level of noise) 
plt.figure()
plt.hist(data_histograms.flatten(), bins = 100, range = (0, 1000))

In [ ]:
phasors_unc = brighteyes_flim.phasor(h5_dataset_sum)
brighteyes_flim.plot_phasor(phasors_unc, bins_2dplot=200, log_scale=False, quadrant='all', dfd_freq = 80e6)

In [ ]:
phasors_pix_apr = phasors_unc / phasor_irf
brighteyes_flim.plot_phasor(phasors_pix_apr, bins_2dplot=200, log_scale=False, quadrant='first', dfd_freq = 80e6)

In [ ]:
tau_phi_apr = brighteyes_flim.calculate_tau_phi(phasors_pix_apr, dfd_freq = 80e6)
print(tau_phi_apr)

In [ ]:
tau_m_apr = brighteyes_flim.calculate_tau_m(phasors_pix_apr, dfd_freq = 80e6)
print(tau_m_apr)

In [ ]:
tau_data_apr = 1e9*tau_phi_apr.flatten()

plt.figure()
plt.hist(tau_data_apr, range = (-3, 8), bins = 50)

In [ ]:
tau_m_data_apr = 1e9*tau_m_apr.flatten()

plt.figure()
plt.hist(tau_m_data_apr, range = (0, 13), bins = 50)

In [ ]:
fig = plt.figure(figsize = (9, 6))
gs = fig.add_gridspec(4, 4)
ax1 = fig.add_subplot(gs[0:2, 0:2])
ax2 = fig.add_subplot(gs[2:4, 0:2])
brighteyes_flim.plot_phasor(phasors_pix_apr[:], bins_2dplot=200, log_scale=False, quadrant='first', fig = fig, ax = ax1, dfd_freq = 80e6)
gr.show_flim(data_histograms, tau_phi_apr*1e9, pxsize = 0.167, pxdwelltime = 500, lifetime_bounds = (1.3, 4.2), intensity_bounds = (0, 1000), fig = fig, ax = ax2)  
fig.tight_layout()
plt.savefig(r"C:\Users\REPLACE_ME\Desktop\PDF_processed_images\APR_convallaria_29_07.pdf", dpi = 900)

### CALCULATE THE LIFETIME WITH THE FITTING (Confocal case)

### Check the global histogram (sum of the decay histograms of each pixel)

In [ ]:
hist_conv = np.sum(data_3D, axis = (0,1))
plt.figure()
plt.plot(hist_conv)

### Define the model

In [ ]:
def exp_fun(A, tau, t):
    decay = A * np.exp(-t/tau) 
   # decay[t<0] = 0
    return decay

def model_2(t, A, tau, bkg):
   
    dt = 0.048 #ns
    nbin = 260 
    period = dt * nbin
    
    irf = final_irf.copy().astype(np.float64)
    irf/= irf.sum()
    decay = (np.heaviside(t, 0) + 1/(np.exp(period/tau) -1)) * exp_fun(A,tau,t)
    decay_convolved = scipy.signal.convolve(decay, irf, mode='same')
    decay_convolved =  decay_convolved[nbin//2:-nbin//2]
    return decay_convolved + bkg

In [ ]:
def fit_function_2(t, A, tau, bkg):
    return model_2(t, A, tau, bkg)

### find the maximum of the histogram generated by the model 

In [ ]:
dt = 0.048 #ns
nbin = 260 
period = dt * nbin
t=np.linspace(-period, period, nbin*2)
t_1 = np.linspace(0, period, nbin) 

decay = (np.heaviside(t, 0) + 1/(np.exp(period/2.5) -1)) * exp_fun(10,2.5,t)
decay_convolved = scipy.signal.convolve(decay, final_irf, mode='same')
decay_convolved =  decay_convolved[nbin//2:-nbin//2]

plt.figure()
plt.plot(t_1, decay_convolved)

In [ ]:
print(np.argmax(decay_convolved))
print(np.argmax(final_irf))
print(np.argmax(hist_conv))

### Get the decay histogram of each pixel, apply the fitting model and extract the lifetime estimated for each pixel (then save it in a 2D matrix)

In [ ]:
hist_shifted = shift(hist_conv, 3.1, order=1, mode='grid-wrap')
print(np.argmax(hist_shifted))

### test on the global fitting

In [ ]:
dt = 0.048 #ns
nbin = 260 
per = dt*nbin
t_axis = np.linspace(-per, per, nbin*2)
initial_guess = [450000, 2.5, 2000]


# Perform the fit
popt, pcov = scipy.optimize.curve_fit(fit_function_2, t_axis, hist_shifted, p0=initial_guess)

# Extract fitted parameters
fitted_A, fitted_tau, fitted_bkg  = popt

In [ ]:
print(f"Fitted parameters: A = {fitted_A}, tau = {fitted_tau}, Background noise = {fitted_bkg}")

In [ ]:
t = np.linspace(0, 12.5, 260)
plt.figure(figsize=(10, 6))
plt.plot(t, hist_shifted, 'b-', label='measured data')
plt.plot(t, fit_function_2(t_axis, *popt), 'r--', label='Fitted model')
plt.xlabel('Time')
plt.ylabel('Intensity')
plt.legend()
plt.show()

### reconstruct the lifetime in each pixel

In [ ]:
lifetime_map = np.zeros((data_3D.shape[0], data_3D.shape[1]))
background_map = np.zeros((data_3D.shape[0], data_3D.shape[1]))
initial_guess = [10000, 2.5, 500]
dt = 0.048 #ns
nbin = 260 
per = dt*nbin
t_axis = np.linspace(-per, per, nbin*2)

for i in range(data_3D.shape[0]):
    for j in range(data_3D.shape[1]):
       
        decay_histogram = data_3D[i, j, :]
        decay_histogram = shift(decay_histogram, 3.1, order=1, mode='grid-wrap')
        
        try:
            popt, pcov = scipy.optimize.curve_fit(fit_function_2, t_axis, decay_histogram, p0=initial_guess)
            fitted_tau = popt[1]
            fitted_bkg = popt[2]
        
        except RuntimeError:
            fitted_tau = np.nan
        
        
        lifetime_map[i, j] = fitted_tau
        background_map[i, j] = fitted_bkg


In [ ]:
print(lifetime_map.shape)

### Compare the distributions of lifetime in each pixel with the tbree methods considered to calculate tau:
### 1. Fitting
### 2. Tau_phi
### 3. Tau_m

In [ ]:
plt.figure()
lifetime_flattened = lifetime_map.flatten()
plt.hist(lifetime_flattened, range = (-3, 8), bins = 50)

In [ ]:
tau_fitting_denoised = lifetime_map[hist_indexes[:, 0], hist_indexes[:, 1]]
print(tau_fitting_denoised)
plt.figure()
plt.hist(tau_fitting_denoised, bins = 100, range = (-1, 10))

In [ ]:
tau_phi_denoised = tau_phi[hist_indexes[:, 0], hist_indexes[:, 1]]
print(tau_phi_denoised)
plt.figure()
plt.hist(1e9*tau_phi_denoised, bins = 50, range = (-3, 8))

In [ ]:
plt.figure()
plt.hist(lifetime_flattened_apr, range = (-3, 8), bins = 50)

In [ ]:
tau_m_denoised = tau_m_clean[hist_indexes[:, 0], hist_indexes[:, 1]]
print(tau_m_denoised.shape)
plt.figure()
plt.hist(1e9*tau_m_denoised, bins = 100, range = (-1, 10))

### Compare the lifetime images obtained with the three considered methods for the confocal case
### 1. Fitting
### 2. Tau_phi
### 3. Tau_m

In [ ]:
fig = plt.figure(figsize = (9, 6))
gr.show_flim(data_2D, lifetime_map, pxsize = 0.167, pxdwelltime = 500, lifetime_bounds = (0.1, 3.2), intensity_bounds = (0, 1000), fig = fig)  
fig.tight_layout()
plt.savefig(r"C:\Users\REPLACE_ME\Desktop\PDF_processed_images\Fitting_convallaria_29_07.pdf", dpi = 900)

In [ ]:
fig = plt.figure(figsize = (9, 6))
gs = fig.add_gridspec(4, 4)
ax1 = fig.add_subplot(gs[0:2, 0:2])
ax2 = fig.add_subplot(gs[2:4, 0:2])
brighteyes_flim.plot_phasor(phasor_cleaned, bins_2dplot=200, log_scale=False, quadrant='first', fig = fig, ax = ax1, dfd_freq = 80e6)
gr.show_flim(data_2D, tau_phi*1e9, pxsize = 0.167, pxdwelltime = 500, lifetime_bounds = (1.3, 3.2), intensity_bounds = (0, 1000), fig = fig, ax = ax2)  
fig.tight_layout()
plt.savefig(r"C:\Users\REPLACE_ME\Desktop\PDF_processed_images\Tau_phi_convallaria_29_07.pdf", dpi = 900)

In [ ]:
fig = plt.figure(figsize = (9, 6))
gs = fig.add_gridspec(4, 4)
ax1 = fig.add_subplot(gs[0:2, 0:2])
ax2 = fig.add_subplot(gs[2:4, 0:2])
brighteyes_flim.plot_phasor(phasor_cleaned, bins_2dplot=200, log_scale=False, quadrant='first', fig = fig, ax = ax1, dfd_freq = 80e6)
gr.show_flim(data_2D, tau_m_clean*1e9, pxsize = 0.167, pxdwelltime = 500, lifetime_bounds = (1.3, 3.2), intensity_bounds = (0, 1000), fig = fig, ax = ax2)  
fig.tight_layout()
plt.savefig(r"C:\Users\REPLACE_ME\Desktop\PDF_processed_images\Tau_m_convallaria_29_07.pdf", dpi = 900)

### CALCULATE LIFETIME WITH FITTING (APR case)

### test on global histogram

In [ ]:
global_hist = np.sum(h5_dataset_sum, axis=(0,1))
plt.figure()
plt.plot(global_hist)

In [ ]:
dt = 0.048 #ns
nbin = 260 
period = dt * nbin
t=np.linspace(-period, period, nbin*2)
t_1 = np.linspace(0, period, nbin) 

decay = (np.heaviside(t, 0) + 1/(np.exp(period/2.5) -1)) * exp_fun(10,2.5,t)
decay_convolved = scipy.signal.convolve(decay, final_irf, mode='same')
decay_convolved =  decay_convolved[nbin//2:-nbin//2]

plt.figure()
plt.plot(t_1, decay_convolved)

In [ ]:
print(np.argmax(global_hist))
print(np.argmax(final_irf))
print(np.argmax(hist_conv))

In [ ]:
global_hist_shifted = shift(global_hist, 0.1, order=1, mode='grid-wrap')
print(np.argmax(global_hist_shifted))

In [ ]:
dt_g = 0.048 #ns
nbin_g = 260 
per_g = dt*nbin
t_axis_g = np.linspace(-per_g, per_g, nbin_g*2)
initial_guess_g = [300000, 2.3, 2000]


# Perform the fit
popt_g, pcov_g = scipy.optimize.curve_fit(fit_function_2, t_axis_g, global_hist_shifted, p0=initial_guess_g)

# Extract fitted parameters
fitted_A_g, fitted_tau_g, fitted_bkg_g  = popt_g

In [ ]:
print(f"Fitted parameters: A = {fitted_A_g}, tau = {fitted_tau_g}, Background noise = {fitted_bkg_g}")

In [ ]:
t = np.linspace(0, 12.5, 260)
plt.figure(figsize=(10, 6))
plt.plot(t, global_hist_shifted, 'b-', label='measured data')
plt.plot(t, fit_function_2(t_axis_g, *popt_g), 'r--', label='Fitted model')
plt.xlabel('Time')
plt.ylabel('Intensity')
plt.legend()
plt.show()

### Reconstruct the lifetime in each pixel (APR dataset)

In [ ]:
lifetime_map_apr = np.zeros((h5_dataset_sum.shape[0], h5_dataset_sum.shape[1]))
initial_guess_apr = [1000, 2.5, 50]
dt = 0.048 #ns
nbin = 260 
per = dt*nbin
t_axis = np.linspace(-per, per, nbin*2)

for i in range(h5_dataset_sum.shape[0]):
    for j in range(h5_dataset_sum.shape[1]):
       
        decay_histogram_apr = h5_dataset_sum[i, j, :]
        decay_histogram_apr = shift(decay_histogram_apr, 3.1, order=1, mode='grid-wrap')
        
        try:
            popt, pcov = scipy.optimize.curve_fit(fit_function_2, t_axis, decay_histogram_apr, p0=initial_guess_apr)
            fitted_tau_apr = popt[1]
            
        except RuntimeError:
            fitted_tau_apr = np.nan
        
        
        lifetime_map_apr[i, j] = fitted_tau_apr

In [ ]:
plt.figure()
lifetime_flattened_apr = lifetime_map_apr.flatten()
plt.hist(lifetime_flattened_apr, range = (-3, 8), bins = 50)

### Fitting: confocal vs APR

In [ ]:
fig = plt.figure(figsize = (9, 6))
gr.show_flim(data_histograms, lifetime_map_apr, pxsize = 0.167, pxdwelltime = 500, lifetime_bounds = (1.3, 3.2), intensity_bounds = (0, 1200), fig = fig)  
fig.tight_layout()
plt.savefig(r"C:\Users\REPLACE_ME\Desktop\PDF_processed_images\Fitting_APR_convallaria_29_07.pdf", dpi = 900)

In [ ]:
fig = plt.figure(figsize = (9, 6))
gr.show_flim(data_2D, lifetime_map, pxsize = 0.167, pxdwelltime = 500, lifetime_bounds = (1.3, 3.2), intensity_bounds = (0, 1200), fig = fig)  
fig.tight_layout()
plt.savefig(r"C:\Users\REPLACE_ME\Desktop\PDF_processed_images\Fitting_convallaria_29_07.pdf", dpi = 900)

In [ ]:
fig = plt.figure(figsize = (9, 6))
gs = fig.add_gridspec(4, 4)
ax1 = fig.add_subplot(gs[0:2, 0:2])
ax2 = fig.add_subplot(gs[2:4, 0:2])
gr.show_flim(data_2D, lifetime_map, pxsize = 0.167, pxdwelltime = 500, lifetime_bounds = (1.3, 3.2), intensity_bounds = (0, 1000), fig = fig, ax = ax1) 
gr.show_flim(data_histograms, lifetime_map_apr, pxsize = 0.167, pxdwelltime = 500, lifetime_bounds = (1.3, 3.2), intensity_bounds = (0, 1000), fig = fig, ax=ax2)
fig.tight_layout()
plt.savefig(r"C:\Users\REPLACE_ME\Desktop\PDF_processed_images\Fitting_Confocal_vs_Fitting_APR_convallaria_29_07.pdf", dpi = 900)

### Tau_m: Confocal vs APR

In [ ]:
fig = plt.figure(figsize = (9, 6))
gr.show_flim(data_histograms, tau_phi_apr*1e9, pxsize = 0.167, pxdwelltime = 500, lifetime_bounds = (1.3, 3.2), intensity_bounds = (0, 1200), fig = fig)  
fig.tight_layout()

In [ ]:
fig = plt.figure(figsize = (9, 6))
gs = fig.add_gridspec(4, 4)
ax1 = fig.add_subplot(gs[0:2, 0:2])
ax2 = fig.add_subplot(gs[2:4, 0:2])
gr.show_flim(data_2D, tau_m_clean*1e9, pxsize = 0.167, pxdwelltime = 500, lifetime_bounds = (1.3, 3.2), intensity_bounds = (0, 1000), fig = fig, ax = ax1)
gr.show_flim(data_histograms, tau_m_apr*1e9, pxsize = 0.167, pxdwelltime = 500, lifetime_bounds = (1.3, 3.2), intensity_bounds = (0, 1000), fig = fig, ax = ax2)  
fig.tight_layout()
plt.savefig(r"C:\Users\REPLACE_ME\Desktop\PDF_processed_images\TauM_Confocal_vs_TauM_APR_convallaria_29_07.pdf", dpi = 900)

### Tau_phi: Confocal vs APR

In [ ]:
fig = plt.figure(figsize = (9, 6))
gs = fig.add_gridspec(4, 4)
ax1 = fig.add_subplot(gs[0:2, 0:2])
ax2 = fig.add_subplot(gs[2:4, 0:2])
gr.show_flim(data_2D, tau_phi*1e9, pxsize = 0.167, pxdwelltime = 500, lifetime_bounds = (1.3, 3.2), intensity_bounds = (0, 1000), fig = fig, ax = ax1)  
gr.show_flim(data_histograms, tau_phi_apr*1e9, pxsize = 0.167, pxdwelltime = 500, lifetime_bounds = (1.3, 3.2), intensity_bounds = (0, 1000), fig = fig, ax = ax2)  
fig.tight_layout()
plt.savefig(r"C:\Users\REPLACE_ME\Desktop\PDF_processed_images\TauPhi_Confocal_vs_TauPhi_APR_convallaria_29_07.pdf", dpi = 900)

In [ ]:
difference_image = lifetime_map_apr - (tau_phi_apr*1e9)

print(difference_image.flatten())

In [ ]:
print(lifetime_map_apr)

In [ ]:
print(tau_phi_apr)

In [ ]:
plt.figure()
tau_diff_flat = difference_image.flatten()
plt.hist(tau_diff_flat, range = (-3, 2), bins = 50)

In [ ]:
plt.figure()
plt.hist(lifetime_flattened_apr, range = (-3, 8), bins = 50)

In [ ]:
fig = plt.figure(figsize = (9, 6))

gr.show_flim(data_2D, difference_image, pxsize = 0.167, pxdwelltime = 500, lifetime_bounds = (0, 0.5), intensity_bounds = (0, 1000), fig = fig)  

fig.tight_layout()
plt.savefig(r"C:\Users\REPLACE_ME\Desktop\PDF_processed_images\Difference_fitting_tau_phi_APR_convallaria_29_07.pdf", dpi = 900)

In [ ]:
plt.figure()
from matplotlib_scalebar.scalebar import ScaleBar
c= plt.imshow(difference_image, vmin=-3, vmax=0.5, cmap='viridis')
plt.colorbar(c) 
  

In [ ]:
plt.figure()

c= plt.imshow(tau_phi*1e9, vmin=0, vmax=3.2, cmap='viridis')
plt.colorbar()
plt.show

In [ ]:
plt.figure()

c= plt.imshow(lifetime_map, vmin=0, vmax=3.2, cmap='viridis')
plt.colorbar()
plt.show

In [ ]:
print(tau_phi.shape)

### show APR

In [ ]:

pxsizex = 0.167
N = 25
usf = 100
ref = N//2

shift, img_ism = apr.APR(image_3d, usf, ref, pxsize = 2*pxsizex)

img_ism_sum = img_ism.sum(axis=-1)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(20,10))

gr.ShowImg(data_2D, pxsize_x = pxsizex, vmin = 0, vmax = 5000, fig = fig, ax = ax[0])
ax[0].set_title('Sum')

gr.ShowImg(img_ism_sum, pxsize_x = pxsizex, vmin = 0, vmax = 5000, fig = fig, ax = ax[1])
ax[1].set_title('APR')



fig.tight_layout()